# 06 — Projeto → votação → decisão → norma

Uma aprovação na votação não prova conclusão na Casa; uma proposição afetada não é necessariamente o objeto votado. Inspecionar exemplos com e sem vínculos e preservar incertezas do registro oficial.

Referência: [limitações documentadas da Câmara](https://dadosabertos.camara.leg.br/howtouse/2020-02-07-dados-votacoes.html).

In [1]:
from pathlib import Path
import sys
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'validation_utils.py').exists())
sys.path.insert(0, str(root / 'scripts'))
from validation_utils import (Probe, OUT, CHECKS, MANIFEST, MANIFEST_FIELDS, api_camara, api_senado,
                              record, manifest_rows, save_manifest, rows_camara, count_values, csv_head)
pd.set_option('display.max_colwidth', 90)

from collections import Counter
probe=Probe('06',max_calls=100)
manifest=manifest_rows()
assert manifest, 'Execute 03 primeiro.'

## Câmara: projeto sem vínculo e votação com objeto/afetados

In [2]:
projects=[x for x in manifest if x['fonte']=='camara' and x['entidade']=='projeto'][:6]
projects += [{'id':'117992','tipo':'caso sem voto amostrado','fonte':'camara'}]
counts=[]
for p in projects:
    r=probe.json(api_camara(f"proposicoes/{p['id']}/votacoes"))
    counts.append({'id_projeto':p['id'],'tipo':p['tipo'],'votacoes_vinculadas':len(rows_camara(r.data)) if r.ok else None,'erro':r.error})
display(pd.DataFrame(counts))
vote=probe.json(api_camara('votacoes/2611313-34'))
d=(vote.data or {}).get('dados',{}) if vote.ok else {}
print('Campos do detalhe:',list(d))
print('Objeto/afetadas:',{k:len(v) if isinstance(v,list) else bool(v) for k,v in d.items() if 'propos' in k.lower() or 'objeto' in k.lower()})
record('06','vinculo_camara','camara','/proposicoes/{id}/votacoes + /votacoes/{id}',len(counts),
       'Distinguir objeto possível e proposições afetadas; não inferir inexistência de voto de lista vazia',
       {'projetos':len(counts),'com_vinculo':sum((x['votacoes_vinculadas'] or 0)>0 for x in counts),
        'campos_detalhe':list(d)},'INCONCLUSIVO','Cobertura e objeto real exigem comparação com arquivos e ficha de tramitação')

,id_projeto,tipo,votacoes_vinculadas,erro
0,2647384,PL,0,
1,2647345,PL,0,
2,2599994,PLP,0,
3,2600175,PLP,0,
4,2604173,PEC,2,
5,2623400,PEC,0,
6,117992,caso sem voto amostrado,0,


Campos do detalhe: ['id', 'uri', 'data', 'dataHoraRegistro', 'siglaOrgao', 'uriOrgao', 'idOrgao', 'uriEvento', 'idEvento', 'descricao', 'aprovacao', 'descUltimaAberturaVotacao', 'dataHoraUltimaAberturaVotacao', 'ultimaApresentacaoProposicao', 'efeitosRegistrados', 'objetosPossiveis', 'proposicoesAfetadas']
Objeto/afetadas: {'ultimaApresentacaoProposicao': True, 'objetosPossiveis': 25, 'proposicoesAfetadas': 1}


{'notebook': '06',
 'check': 'vinculo_camara',
 'fonte': 'camara',
 'url_parametros': '/proposicoes/{id}/votacoes + /votacoes/{id}',
 'coletado_em_utc': '2026-10-09T14:23:15+00:00',
 'amostra_n': 7,
 'regra': 'Distinguir objeto possível e proposições afetadas; não inferir inexistência de voto de lista vazia',
 'contagens': '{"campos_detalhe": ["id", "uri", "data", "dataHoraRegistro", "siglaOrgao", "uriOrgao", "idOrgao", "uriEvento", "idEvento", "descricao", "aprovacao", "descUltimaAberturaVotacao", "dataHoraUltimaAberturaVotacao", "ultimaApresentacaoProposicao", "efeitosRegistrados", "objetosPossiveis", "proposicoesAfetadas"], "com_vinculo": 1, "projetos": 7}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Cobertura e objeto real exigem comparação com arquivos e ficha de tramitação'}

## Arquivos de objetos possíveis e proposições afetadas

In [3]:
base='https://dadosabertos.camara.leg.br/arquivos'
names=('votacoesObjetos','votacoesProposicoes')
results=[]
for name in names:
    url=f'{base}/{name}/csv/{name}-2025.csv'
    r=probe.fetch(url,limit=80_000_000)
    sample=csv_head(r.content) if r.ok else []
    results.append({'relação':name,'HTTP':r.status,'URL':r.url,'bytes':len(r.content or b''),
                    'colunas':list(sample[0]) if sample else [],'erro':r.error})
display(pd.DataFrame(results))
record('06','arquivos_relacao_votacao','camara',base,len(results),
       'Confirmar nomes, cabeçalhos e disponibilidade dos dois conjuntos distintos',
       {'arquivos':{x['relação']:{'HTTP':x['HTTP'],'colunas':x['colunas'],'erro':x['erro']} for x in results}},
       'PASSOU' if all(x['HTTP']==200 and x['colunas'] for x in results) else 'INCONCLUSIVO',
       'Documentação antiga cita outro nome; usar o nome efetivo do portal após conferir cabeçalhos')

,relação,HTTP,URL,bytes,colunas,erro
0,votacoesObjetos,200,https://dadosabertos.camara.leg.br/arquivos/votacoesObjetos/csv/votacoesObjetos-2025.csv,45036655,"[idVotacao, uriVotacao, data, descricao, proposicao_id, proposicao_uri, proposicao_eme...",
1,votacoesProposicoes,200,https://dadosabertos.camara.leg.br/arquivos/votacoesProposicoes/csv/votacoesProposicoe...,4997496,"[idVotacao, uriVotacao, data, descricao, proposicao_id, proposicao_uri, proposicao_tit...",


{'notebook': '06',
 'check': 'arquivos_relacao_votacao',
 'fonte': 'camara',
 'url_parametros': 'https://dadosabertos.camara.leg.br/arquivos',
 'coletado_em_utc': '2026-10-09T14:23:17+00:00',
 'amostra_n': 2,
 'regra': 'Confirmar nomes, cabeçalhos e disponibilidade dos dois conjuntos distintos',
 'contagens': '{"arquivos": {"votacoesObjetos": {"HTTP": 200, "colunas": ["idVotacao", "uriVotacao", "data", "descricao", "proposicao_id", "proposicao_uri", "proposicao_ementa", "proposicao_codTipo", "proposicao_siglaTipo", "proposicao_numero", "proposicao_ano", "proposicao_titulo"], "erro": ""}, "votacoesProposicoes": {"HTTP": 200, "colunas": ["idVotacao", "uriVotacao", "data", "descricao", "proposicao_id", "proposicao_uri", "proposicao_titulo", "proposicao_ementa", "proposicao_codTipo", "proposicao_siglaTipo", "proposicao_numero", "proposicao_ano"], "erro": ""}}}',
 'estado': 'PASSOU',
 'motivo': 'Documentação antiga cita outro nome; usar o nome efetivo do portal após conferir cabeçalhos'}

## Senado: idProcesso, voto, tramitação e normaGerada

In [4]:
v=probe.json(api_senado('votacao.json'),{'codigoParlamentar':'5672','dataInicio':'2023-02-01','dataFim':'2023-02-28'})
votes=v.data if v.ok and isinstance(v.data,list) else []
links=[]
for vote in votes[:3]:
    pid=vote.get('idProcesso')
    p=probe.json(api_senado(f'processo/{pid}.json')) if pid else None
    links.append({'idProcesso_votacao':pid,'id_detalhe':p.data.get('id') if p and p.ok else None,
                  'codigoMateria_votacao':vote.get('codigoMateria'),
                  'codigoMateria_detalhe':p.data.get('codigoMateria') if p and p.ok else None})
display(pd.DataFrame(links))
norm=probe.json(api_senado('processo/9105948.json'))
norma=norm.data.get('normaGerada') if norm.ok else None
display(pd.DataFrame([{'processo':9105948,'norma_tipo':(norma or {}).get('siglaTipo'),
                      'numero':(norma or {}).get('numero'),'publicacao':(norma or {}).get('dataPublicacao')}]))
record('06','vinculo_e_norma_senado','senado','/votacao.json + /processo/{id}.json',len(links)+1,
       'idProcesso resolve detalhe; normaGerada é etapa separada da votação',
       {'vinculos_confirmados':sum(x['idProcesso_votacao']==x['id_detalhe'] for x in links),
        'norma_exemplo':bool(norma)},'INCONCLUSIVO',
       'Um caso positivo não estabelece cobertura nem regra de conclusão na Casa; '+v.error+' '+norm.error)

,idProcesso_votacao,id_detalhe,codigoMateria_votacao,codigoMateria_detalhe
0,8361684,8361684,155773,155773


,processo,norma_tipo,numero,publicacao
0,9105948,LCP,237,2026-09-15


{'notebook': '06',
 'check': 'vinculo_e_norma_senado',
 'fonte': 'senado',
 'url_parametros': '/votacao.json + /processo/{id}.json',
 'coletado_em_utc': '2026-10-09T14:23:20+00:00',
 'amostra_n': 2,
 'regra': 'idProcesso resolve detalhe; normaGerada é etapa separada da votação',
 'contagens': '{"norma_exemplo": true, "vinculos_confirmados": 1}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Um caso positivo não estabelece cobertura nem regra de conclusão na Casa;  '}

## Gate semântico: aprovação específica, conclusão na Casa e publicação

In [5]:
for check,rule in [('aprovacao_votacao','Resultado válido somente para a decisão específica'),
                   ('conclusao_casa','Vocabulários e tramitação final por Casa'),
                   ('norma_publicada','Tipo de norma, publicação e vínculo de origem')]:
    record('06',check,'ambas','detalhes e tramitações amostrados',len(counts)+len(links),rule,{},
           'INCONCLUSIVO','Não gerar taxa ou funil público até validar regra e cobertura por Casa')
print('Três etapas mantidas separadas. Nenhuma taxa de aprovação calculada.')

Três etapas mantidas separadas. Nenhuma taxa de aprovação calculada.
